In [7]:
import numpy as np
import os
import pandas as pd
import re
from pathlib import Path
import pyreadr

In [8]:
DATA_DIR = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data")

In [9]:
# ---- Load pre-built metadata instead of rebuilding ----
meta_all = pd.read_csv(DATA_DIR / "training_samples_meta.csv", parse_dates=["event_datetime", "t0_datetime", "target_datetime_10", "target_datetime_20", "target_datetime_30"])
print("Loaded meta_all:", meta_all.shape)
print("Unique events:", meta_all["event_id"].nunique())

# ---- Load arrays ----
data = np.load(DATA_DIR / "training_data.npz")
X_all = data["X"]  # (8602, 3, 501, 371) float32
Y_all = data["Y"]  # (8602, 3, 501, 371) float32
print("X_all:", X_all.shape)
print("Y_all:", Y_all.shape)


Loaded meta_all: (8602, 12)
Unique events: 253
X_all: (8602, 3, 501, 371)
Y_all: (8602, 3, 501, 371)


In [10]:
# derive stats ----
Y_flat = Y_all.reshape(Y_all.shape[0], -1)

meta_all["mean_intensity"] = Y_flat.mean(axis=1)
meta_all["max_intensity"]  = Y_flat.max(axis=1)

wet_mask = Y_flat > 0.1
meta_all["mean_intensity_wet"] = [
    Y_flat[i][wet_mask[i]].mean() if wet_mask[i].any() else 0.0
    for i in range(Y_flat.shape[0])
]
meta_all["wet_fraction"] = wet_mask.sum(axis=1) / Y_flat.shape[1]

# ---- build split ----
events = (
    meta_all[["event_id", "event_datetime", "season"]]
    .drop_duplicates()
    .sort_values(["season", "event_datetime"])
)


In [11]:
### Define operational split ###

train_events, val_events, test_events = [], [], []
for season, df in events.groupby("season"):
    df = df.sort_values("event_datetime")
    n = len(df)
    train_end = int(0.70 * n)
    val_end   = int(0.85 * n)
    train_events.extend(df.iloc[:train_end]["event_id"])
    val_events.extend(df.iloc[train_end:val_end]["event_id"])
    test_events.extend(df.iloc[val_end:]["event_id"])

meta_all["split"] = "train"
meta_all.loc[meta_all["event_id"].isin(val_events),  "split"] = "val"
meta_all.loc[meta_all["event_id"].isin(test_events), "split"] = "test"

print(meta_all["split"].value_counts())

train_mask = meta_all["split"] == "train"
val_mask   = meta_all["split"] == "val"
test_mask  = meta_all["split"] == "test"

split
train    5984
test     1360
val      1258
Name: count, dtype: int64


In [12]:
# ---- Log transform ----
X_all_log = np.log1p(X_all).astype(np.float32)
Y_all_log = np.log1p(Y_all).astype(np.float32)

# ---- Final splits ----
train_mask = meta_all["split"] == "train"
val_mask   = meta_all["split"] == "val"
test_mask  = meta_all["split"] == "test"

X_train, Y_train = X_all_log[train_mask], Y_all_log[train_mask]
X_val,   Y_val   = X_all_log[val_mask],   Y_all_log[val_mask]
X_test,  Y_test  = X_all_log[test_mask],  Y_all_log[test_mask]

meta_train = meta_all.loc[train_mask].reset_index(drop=True)
meta_val   = meta_all.loc[val_mask].reset_index(drop=True)
meta_test  = meta_all.loc[test_mask].reset_index(drop=True)

print(f"X_train: {X_train.shape}  Y_train: {Y_train.shape}")
print(f"X_val:   {X_val.shape}  Y_val:   {Y_val.shape}")
print(f"X_test:  {X_test.shape}  Y_test:  {Y_test.shape}")


X_train: (5984, 3, 501, 371)  Y_train: (5984, 3, 501, 371)
X_val:   (1258, 3, 501, 371)  Y_val:   (1258, 3, 501, 371)
X_test:  (1360, 3, 501, 371)  Y_test:  (1360, 3, 501, 371)


In [13]:
# ---- Persistence baseline (in original space) ----
LEAD_NAMES = ["+10min", "+20min", "+30min"]
Y_persist = X_all[test_mask][:, -1:, :, :]   # (N_test, 1, H, W) — last input frame

mse_persist = {}
mae_persist = {}
for j, name in enumerate(LEAD_NAMES):
    Y_true_j = Y_all[test_mask][:, j:j+1, :, :]
    mse_persist[name] = float(np.mean((Y_persist - Y_true_j)**2))
    mae_persist[name] = float(np.mean(np.abs(Y_persist - Y_true_j)))
    print(f"Persistence {name}  MSE: {mse_persist[name]:.6f}  MAE: {mae_persist[name]:.6f}")


Persistence +10min  MSE: 0.135091  MAE: 0.051614
Persistence +20min  MSE: 0.216263  MAE: 0.073027
Persistence +30min  MSE: 0.237859  MAE: 0.082429


In [14]:
# ---- Persistence CSI @ multiple thresholds ----
thresholds = [0.1, 0.5, 1.0]
csi_results = {}  # keyed by (lead_name, threshold)

for j, name in enumerate(LEAD_NAMES):
    Y_true_j = Y_all[test_mask][:, j:j+1, :, :]
    for thr in thresholds:
        tp  = np.sum((Y_persist >= thr) & (Y_true_j >= thr))
        fp  = np.sum((Y_persist >= thr) & (Y_true_j <  thr))
        fn  = np.sum((Y_persist <  thr) & (Y_true_j >= thr))
        csi = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float("nan")
        csi_results[(name, thr)] = csi
        print(f"Persistence {name}  CSI @{thr}mm/10min: {csi:.4f}")


Persistence +10min  CSI @0.1mm/10min: 0.6805
Persistence +10min  CSI @0.5mm/10min: 0.4736
Persistence +10min  CSI @1.0mm/10min: 0.3413
Persistence +20min  CSI @0.1mm/10min: 0.5452
Persistence +20min  CSI @0.5mm/10min: 0.3101
Persistence +20min  CSI @1.0mm/10min: 0.1774
Persistence +30min  CSI @0.1mm/10min: 0.4721
Persistence +30min  CSI @0.5mm/10min: 0.2412
Persistence +30min  CSI @1.0mm/10min: 0.1254


In [15]:
from scipy.ndimage import uniform_filter

def fss_score(pred, obs, threshold, scale_px):
    pred_bin  = (pred >= threshold).astype(np.float32)
    obs_bin   = (obs  >= threshold).astype(np.float32)
    size      = 2 * scale_px + 1
    pred_frac = uniform_filter(pred_bin, size=size, mode="constant")
    obs_frac  = uniform_filter(obs_bin,  size=size, mode="constant")
    fbs       = np.mean((pred_frac - obs_frac) ** 2)
    fbs_worst = np.mean(pred_frac ** 2) + np.mean(obs_frac ** 2)
    return 1.0 - fbs / fbs_worst if fbs_worst > 0 else float("nan")

FSS_THRESHOLDS = [0.1, 0.5, 1.0]
FSS_SCALES_PX  = [8, 32]

rows = []
for j, name in enumerate(LEAD_NAMES):
    Y_true_j = Y_all[test_mask][:, j, :, :]  # (N, H, W)
    rows += [
        {"model": "persistence", "lead": name, "metric": "mse",  "threshold_mm": None, "scale_px": None, "value": mse_persist[name]},
        {"model": "persistence", "lead": name, "metric": "mae",  "threshold_mm": None, "scale_px": None, "value": mae_persist[name]},
    ]
    rows += [{"model": "persistence", "lead": name, "metric": "csi", "threshold_mm": thr, "scale_px": None,
              "value": csi_results[(name, thr)]} for thr in [0.1, 0.5, 1.0]]
    rows += [{"model": "persistence", "lead": name, "metric": "fss", "threshold_mm": thr, "scale_px": s,
              "value": np.nanmean([fss_score(Y_persist[i, 0], Y_true_j[i], thr, s) for i in range(len(Y_true_j))])}
             for thr in FSS_THRESHOLDS for s in FSS_SCALES_PX]

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))
df_metrics.to_csv(DATA_DIR / "persistence_metrics.csv", index=False)


      model   lead metric  threshold_mm  scale_px    value
persistence +10min    mse           NaN       NaN 0.135091
persistence +10min    mae           NaN       NaN 0.051614
persistence +10min    csi           0.1       NaN 0.680454
persistence +10min    csi           0.5       NaN 0.473569
persistence +10min    csi           1.0       NaN 0.341327
persistence +10min    fss           0.1       8.0 0.934006
persistence +10min    fss           0.1      32.0 0.981060
persistence +10min    fss           0.5       8.0 0.832401
persistence +10min    fss           0.5      32.0 0.934282
persistence +10min    fss           1.0       8.0 0.706364
persistence +10min    fss           1.0      32.0 0.833456
persistence +20min    mse           NaN       NaN 0.216263
persistence +20min    mae           NaN       NaN 0.073027
persistence +20min    csi           0.1       NaN 0.545166
persistence +20min    csi           0.5       NaN 0.310082
persistence +20min    csi           1.0       NaN 0.1774

In [16]:
### Save train/val/test split ###
meta_all.to_csv(DATA_DIR / "training_samples_meta_enriched.csv", index=False)
print("Saved enriched metadata with split column.")


Saved enriched metadata with split column.
